In [ ]:
import pandas as pd
df = pd.read_json("../data/raw/test_run_v1.jsonl", lines=True)

Check the total number of matches in the dataset.

In [2]:
len(df['match_id'])

4000

In [4]:
df['match_id'].nunique()

4000

The number of unique match IDs equals the total number of rows, so there are no duplicate matches.

In [5]:
df['start_dt'] = pd.to_datetime(df['start_time'], unit='s')

In [8]:
df['start_dt'].max(), df['start_dt'].min()

(Timestamp('2026-10-12 07:30:51'), Timestamp('2026-10-11 17:42:53'))

In [10]:
df['start_dt'].dt.floor('10min').value_counts().sort_index()

start_dt
2026-10-11 17:40:00    1509
2026-10-11 17:50:00     476
2026-10-11 18:00:00      11
2026-10-11 18:10:00       4
2026-10-12 06:40:00      67
2026-10-12 06:50:00    1353
2026-10-12 07:00:00     521
2026-10-12 07:10:00      55
2026-10-12 07:20:00       3
2026-10-12 07:30:00       1
Name: count, dtype: int64

In [16]:
df['duration_min'] = df['duration'] / 60
df.loc[df['duration'] > 0, 'duration_min'].describe()

count    3960.000000
mean       22.636507
std         5.543345
min         0.033333
25%        19.300000
50%        22.183333
75%        25.916667
max        39.000000
Name: duration_min, dtype: float64

After two rounds of data collection, we see that 2,000 matches per session is not enough: the sample currently contains only short games.

In [17]:
ranked = df[(df['lobby_type'] == 7) & (df['game_mode'] == 22)]
len(ranked), ranked['duration_min'].describe()

(624,
 count    624.000000
 mean      27.430929
 std        6.581744
 min        1.383333
 25%       24.441667
 50%       28.291667
 75%       32.020833
 max       38.116667
 Name: duration_min, dtype: float64)

## Summary of the test collection

- **Ranked share:** 624 of 4,000 matches (15.6%) are ranked All Draft (`lobby_type = 7`, `game_mode = 22`). The rest are mostly Turbo, bot games and new-player matches, which we exclude.
- **Empty records:** 40 matches (1%) have `duration = 0` and no heroes. They are invalid and will be removed.
- **Sampling bias towards short games:** each collection run covered only ~20–30 minutes of match starts right before the moment of collection. A match appears in the API only after it ends, so only games that had already finished were captured: the maximum duration is 39 min (38 min for ranked), roughly equal to the age of the oldest match in the window. Long games are systematically missing.
- **Implication:** collecting more matches the same way would dilute the bias but not remove it — the first ~1–1.5 hours before every run stay biased.
- **Next step (hypothesis to verify):** start collection from a point at least 3 hours in the past, where all matches should have finished. To confirm, check that the median duration is stable across 10-minute start intervals and that games longer than 40 minutes appear.